# 13. Missing Values

Handling missing data is one of the most important steps in data cleaning. Pandas provides several functions to detect, remove, or fill missing values.


## Setup

Let's create a sample **customers** DataFrame that contains some missing values, similar to the data used in earlier notebooks.

In [1]:
import pandas as pd
import numpy as np
customers = pd.DataFrame({
    "Name": ["Hema", "Chitra", "Koushi", "Subha", "Swathi"],
    "Age": [25, np.nan, 30, 28, np.nan],
    "City": ["Chennai", "Delhi", np.nan, "Mumbai", "Chennai"],
    "Salary": [50000, 55000, np.nan, np.nan, 62000]
})
customers

,Name,Age,City,Salary
0,Hema,25.0,Chennai,50000.0
1,Chitra,NaN,Delhi,55000.0
2,Koushi,30.0,NaN,NaN
3,Subha,28.0,Mumbai,NaN
4,Swathi,NaN,Chennai,62000.0


## 1. isnull()

**isnull()** checks each value in a Series or DataFrame and returns **True** where the value is missing (NaN), and **False** otherwise.

**Syntax:**

```
DataFrame.isnull()
Series.isnull()
```

In [2]:
customers.isnull()

,Name,Age,City,Salary
0,False,False,False,False
1,False,True,False,False
2,False,False,True,True
3,False,False,False,True
4,False,True,False,False


Example:

A company may want to quickly find out which employee records have missing salary information.

In [3]:
customers["Salary"].isnull()

0    False
1    False
2     True
3     True
4    False
Name: Salary, dtype: bool

### AI/ML Usage

Detecting missing values is the first step before any imputation or model training, since most ML algorithms cannot handle NaN values directly.

## 2. notnull()

**notnull()** is the exact opposite of **isnull()**. It returns **True** where a value is present and **False** where it is missing.

**Syntax:**

```
DataFrame.notnull()
Series.notnull()
```

In [4]:
customers.notnull()

,Name,Age,City,Salary
0,True,True,True,True
1,True,False,True,True
2,True,True,False,False
3,True,True,True,False
4,True,False,True,True


Example:

A company may want to filter out only the customers whose City is known.

In [5]:
customers[customers["City"].notnull()]

,Name,Age,City,Salary
0,Hema,25.0,Chennai,50000.0
1,Chitra,NaN,Delhi,55000.0
3,Subha,28.0,Mumbai,NaN
4,Swathi,NaN,Chennai,62000.0


### AI/ML Usage

Useful for filtering out complete records to build a clean training subset before modeling.

## 3. dropna()

**dropna()** removes rows (or columns) that contain missing values.

**Syntax:**

```
DataFrame.dropna(axis=0, how='any', subset=None)
```

In [6]:
customers.dropna()

,Name,Age,City,Salary
0,Hema,25.0,Chennai,50000.0


Example:

A company may want to drop any customer record where the salary is not available before running payroll analysis.

In [7]:
customers.dropna(subset=["Salary"])

,Name,Age,City,Salary
0,Hema,25.0,Chennai,50000.0
1,Chitra,NaN,Delhi,55000.0
4,Swathi,NaN,Chennai,62000.0


### AI/ML Usage

Simple but effective strategy for small amounts of missing data; however it can lead to loss of valuable data if overused, so it should be applied carefully.

## 4. fillna()

**fillna()** replaces missing values with a specified value, a statistic (like mean/median), or using forward/backward fill.

**Syntax:**

```
DataFrame.fillna(value=None, method=None)
```

In [8]:
customers.fillna({"City": "Unknown"})

,Name,Age,City,Salary
0,Hema,25.0,Chennai,50000.0
1,Chitra,NaN,Delhi,55000.0
2,Koushi,30.0,Unknown,NaN
3,Subha,28.0,Mumbai,NaN
4,Swathi,NaN,Chennai,62000.0


Example:

A company may want to fill missing Age values with the average age of all employees.

In [9]:
customers["Age"].fillna(customers["Age"].mean())

0    25.000000
1    27.666667
2    30.000000
3    28.000000
4    27.666667
Name: Age, dtype: float64

### AI/ML Usage

Mean/median/mode imputation with **fillna()** is one of the most common preprocessing steps used to prepare numeric features for machine learning models.

## 5. interpolate()

**interpolate()** estimates missing values by using the values around them, commonly using linear interpolation by default. This works well for numeric, ordered data such as time series.

**Syntax:**

```
Series.interpolate(method='linear')
```

In [10]:
customers["Salary"].interpolate()

0    50000.000000
1    55000.000000
2    57333.333333
3    59666.666667
4    62000.000000
Name: Salary, dtype: float64

Example:

A company tracking monthly revenue may want to estimate a missing month's revenue based on the trend of the surrounding months.

In [11]:
revenue = pd.Series([100000, np.nan, np.nan, 130000, 140000])
revenue.interpolate()

0    100000.0
1    110000.0
2    120000.0
3    130000.0
4    140000.0
dtype: float64

### AI/ML Usage

Interpolation is especially useful for time-series datasets used in forecasting models, where preserving the trend of the data is more important than using a single fixed replacement value.

## 6. Missing Value Handling Strategies

There is no single "correct" way to handle missing values — the right strategy depends on the dataset, the amount of missing data, and the type of column.

| Strategy | Function | When to Use |
|---|---|---|
| Detect missing values | `isnull()`, `notnull()` | Always, as the first step |
| Remove rows/columns | `dropna()` | When missing data is small in amount or a row/column is mostly empty |
| Replace with a fixed/statistical value | `fillna()` | When missing data is categorical, or numeric with a stable mean/median |
| Estimate using nearby values | `interpolate()` | When data is numeric and ordered, such as time series |

**General guidelines:**
* Always start by checking how much data is missing using **isnull().sum()**.
* If a column has too many missing values (e.g. more than 50%), consider dropping the column entirely.
* Use **fillna()** with mean/median for numeric columns and mode for categorical columns.
* Use **interpolate()** for time-based or sequential numeric data.
* Never blindly drop or fill data — understand *why* the values are missing first.

In [12]:
customers.isnull().sum()

Name      0
Age       2
City      1
Salary    2
dtype: int64

### AI/ML Usage

Choosing the right missing value strategy directly impacts model accuracy. Poor handling of missing data can introduce bias or reduce the amount of useful training data available to the model.